# CCW+Pancake Wrap Torque Analysis - CSC Version

This notebook contains analysis that belongs to [SITCOM-1854 CCW+Pancake Wrap Torque Analysis]. 

Following the comment in the notebook, we are interested in three particular time windows (all in UTC):

* [From 2025-02-25 17:00 to 2025-02-26 13:00] - Besides the torques associated with movements, we can see motor 2 with currents for a very long time (the whole night).  
  This is not part of the analysis but might need investigation.

* [From 2025-02-27 12:30 to 2025-02-27 18:30] - This is when we, in theory, moved the rotator to +/- 90º with different hexapod positions. 

* [From 2025-02-27 23:30 to 2025-02-28 02:00] - This is a short soak test. Analysis for this might be useful to preliminary results.

* [From 2025-02-28 00:01:56 to 01:38:52 UTC]

* [From 2025-02-28 17:38:43 to 17:49:03 UTC]

* [From 2025-02-28 17:57:50 to 18:02:21 UTC]

* [From 2025-03-01 13:35:05 to 15:54:36 UTC]

[SITCOM-1854 CCW+Pancake Wrap Torque Analysis]: https://rubinobs.atlassian.net/browse/SITCOM-1854

[From 2025-02-25 17:00 to 2025-02-26 13:00]: https://usdf-rsp.slac.stanford.edu/chronograf/sources/1/dashboards/134?refresh=Paused&lower=2025-02-25T12%3A00%3A00.000Z&upper=2025-02-26T18%3A00%3A00.000Z&zoomedLower=2025-02-25T17%3A03%3A58.121Z&zoomedUpper=2025-02-26T12%3A56%3A47.624Z

[From 2025-02-27 12:30 to 2025-02-27 18:30]: https://usdf-rsp.slac.stanford.edu/chronograf/sources/1/dashboards/134?refresh=Paused&lower=2025-02-27T12%3A00%3A00.000Z&upper=2025-02-28T02%3A00%3A00.000Z&zoomedLower=2025-02-27T12%3A31%3A01.573Z&zoomedUpper=2025-02-27T18%3A30%3A21.123Z

[From 2025-02-27 23:30 to 2025-02-28 02:00]: https://usdf-rsp.slac.stanford.edu/chronograf/sources/1/dashboards/134?refresh=Paused&lower=2025-02-27T23%3A30%3A00.000Z&upper=2025-02-28T02%3A00%3A00.000Z

[From 2025-02-28 00:01:56 to 01:38:52 UTC]: https://usdf-rsp.slac.stanford.edu/chronograf/sources/1/dashboards/134?refresh=Paused&lower=2025-02-28T00%3A01%3A56Z&upper=2025-02-28T01%3A38%3A52Z

---
## Set up

Let's import packages and define global variables.

In [ ]:
from datetime import timedelta
from matplotlib import pyplot as plt
from matplotlib.dates import DateFormatter

from astropy.time import Time
from pathlib import Path
from datetime import datetime

from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData


efd_client = makeEfdClient()

In [ ]:
class DataSet:
    """Let's wrap our dataset together"""

    def __init__(self, t_start, t_end, df=None):
        self.t_start = Time(t_start, scale="utc")
        self.t_end = Time(t_end, scale="utc")

        self.df = getEfdData(
            begin=self.t_start,
            end=self.t_end,
            client=efd_client,
            topic="lsst.sal.MTMount.cameraCableWrap",
            columns=[
                "actualPosition",
                "actualVelocity",
                "actualTorquePercentage0",
                "actualTorquePercentage1",
            ],
        )
        self.rdf = self.df.resample("0.5s").mean()

    def __str__(self):
        msg = (
            f"Dataset from {self.t_start.iso} to {self.t_end.iso} with {len(self.df.index)} rows.\n"
            f"  Maximum absolute value for Motor 1: {self.df.actualTorquePercentage0.abs().max():.2f} %\n"
            f"  Maximum absolute value for Motor 2: {self.df.actualTorquePercentage1.abs().max():.2f} %\n"
            f"  Chronograf URL:\n"
            f"    {self.chronograf()}"
        )
        return msg

    def chronograf(self):
        lower = self.t_start.isot.replace(":", "%3A")
        upper = self.t_end.isot.replace(":", "%3A")
        url = f"https://usdf-rsp.slac.stanford.edu/chronograf/sources/1/dashboards/134?refresh=Paused&lower={lower}Z&upper={upper}Z"
        return url

In [ ]:
def plot_torque_and_position_vs_time(df, title):
    fig, ax = plt.subplots(num=title)

    ax.plot(df["actualTorquePercentage0"], color="C0", label="Motor 1")
    ax.plot(df["actualTorquePercentage1"], color="C1", label="Motor 2")
    ax.grid(":", alpha=0.25)
    ax.legend(loc="upper right")
    ax.set_xlabel("Time [UTC]")
    ax.set_ylabel("CCW Torques [%]")

    date_form = DateFormatter("%H:%M")
    ax.xaxis.set_major_formatter(date_form)

    ax2 = ax.twinx()
    ax2.plot(df["actualPosition"], color="C2", label="Actual Position")
    ax2.set_ylabel("CCW Angle [deg]")
    ax2.legend(loc="lower right")

    fig.autofmt_xdate()
    fig.suptitle("CCW Torques Percentage")

    plt.show()

In [ ]:
def plot_torque_vs_angle(df, title):
    fig, ax = plt.subplots(num=title)

    ax.scatter(
        df["actualPosition"],
        df["actualTorquePercentage0"],
        marker=".",
        color="C0",
        label="Motor 1",
        alpha=0.5,
    )
    ax.scatter(
        df["actualPosition"],
        df["actualTorquePercentage1"],
        marker=".",
        color="C1",
        label="Motor 2",
        alpha=0.5,
    )

    ax.grid(":", alpha=0.25)
    ax.legend(loc="upper right")
    ax.set_xlabel("Actual Position [deg]")
    ax.set_ylabel("CCW Torques [%]")
    ax.set_xlim(-95, +95)

    fig.autofmt_xdate()
    fig.suptitle("CCW Torques Percentage")

    plt.show()

## From 2025-02-25 17:00Z to 2025-02-26 13:00Z

In [ ]:
ds1 = DataSet(t_start="2025-02-25T17:00:00Z", t_end="2025-02-26T13:00:00Z")

print(ds1)

In [ ]:
%matplotlib inline
plot_torque_and_position_vs_time(ds1.df, "Dataset 1 - CCW Torques and Position Vs Time")
plot_torque_vs_angle(ds1.df, title="Dataset 1 - Torque vs Angle")

## From 2025-02-27 12:30 to 2025-02-27 18:30 

In [ ]:
ds2 = DataSet(t_start="2025-02-27T12:30:00Z", t_end="2025-02-27T18:30:00Z")

print(ds2)

In [ ]:
%matplotlib inline
plot_torque_and_position_vs_time(
    ds2.rdf, "Dataset 2 - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(ds2.rdf, title="Dataset 2 - Torque vs Angle")

## From 2025-02-27 23:30 to 2025-02-28 02:00

In [ ]:
ds3 = DataSet(t_start="2025-02-27T23:30:00Z", t_end="2025-02-28T02:00:00Z")

print(ds3)

In [ ]:
%matplotlib inline
plot_torque_and_position_vs_time(
    ds3.rdf, "Dataset 3 - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(ds3.rdf, title="Dataset 3 - Torque vs Angle")

## From 2025-02-28 00:01:56 to 01:38:52 UTC

In [ ]:
ds4 = DataSet(t_start="2025-02-28T00:01:56Z", t_end="2025-02-28T01:38:52Z")

print(ds4)

In [ ]:
%matplotlib inline
plot_torque_and_position_vs_time(ds4.df, "Dataset 4 - CCW Torques and Position Vs Time")
plot_torque_vs_angle(ds4.df, title="Dataset 4 - Torque vs Angle")

## From 2025-02-28 17:38:43 to 17:49:03 UTC

In [ ]:
ds5 = DataSet(t_start="2025-02-28T17:38:43Z", t_end="2025-02-28T17:49:03Z")

print(ds5)

In [ ]:
%matplotlib inline
plot_torque_and_position_vs_time(ds4.df, "Dataset 5 - CCW Torques and Position Vs Time")
plot_torque_vs_angle(ds5.df, title="Dataset 5 - Torque vs Angle")

## From 2025-02-28 17:57:50 to 18:02:21 UTC

In [ ]:
ds6 = DataSet(t_start="2025-02-28T17:57:50Z", t_end="2025-02-28T18:02:21Z")

print(ds6)

In [ ]:
%matplotlib inline
plot_torque_and_position_vs_time(ds5.df, "Dataset 6 - CCW Torques and Position Vs Time")
plot_torque_vs_angle(ds6.df, title="Dataset 6 - Torque vs Angle")

## From 2025-03-01 13:35:05 to 15:54:36 UTC

In [ ]:
ds7 = DataSet(t_start="2025-03-01T13:35:05Z", t_end="2025-03-01T15:54:36Z")

print(ds7)

In [ ]:
%matplotlib inline
plot_torque_and_position_vs_time(ds7.df, "Dataset 7 - CCW Torques and Position Vs Time")
plot_torque_vs_angle(ds7.df, title="Dataset 7 - Torque vs Angle")

## ComCam Analysis

There is too much data during ComCam campaign to perform the same plots.  
Because of that, let's select the days where we exercised the CCW the most.

In [ ]:
cc_start = Time("2024-10-24T00:00:00", scale="utc", format="isot")
cc_end = Time("2024-12-10T00:00:00", scale="utc", format="isot")

In [ ]:
query = f"""
SELECT
min(actualPosition) as min_pos,
max(actualPosition) as max_pos
FROM "lsst.sal.MTMount.cameraCableWrap"
WHERE time >= '{cc_start.isot}Z'
AND time <= '{cc_end.isot}Z'
GROUP BY time(24h)
"""
campaign_df = await efd_client.influx_client.query(query)

# We want to select the days where the rotator moved near its full limit
mask = (campaign_df.min_pos < -70) & (campaign_df.max_pos > 70)
campaign_df = campaign_df[mask]

In [ ]:
campaign_df

In [ ]:
index = 0

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")

In [ ]:
index = 1

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")

In [ ]:
index = 2

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")

In [ ]:
index = 3

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")

In [ ]:
index = 4

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")

In [ ]:
index = 5

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")

In [ ]:
index = 6

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")

In [ ]:
index = 7

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")

In [ ]:
index = 8

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")

In [ ]:
index = 9

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")

In [ ]:
index = 10

t_start = campaign_df.index[index].tz_convert(None).to_pydatetime()
t_end = t_start + timedelta(days=1)

cc_ds = DataSet(t_start=t_start.isoformat(), t_end=t_end.isoformat())

print(cc_ds)

%matplotlib inline
plot_torque_and_position_vs_time(
    cc_ds.df, f"CC{index} - CCW Torques and Position Vs Time"
)
plot_torque_vs_angle(cc_ds.df, title=f"CC{index} - Torque vs Angle")